In [1]:
"""This file contains the code that is to run the analysis"""
import numpy as np
import orbits
import dynamics
import linearize

In [2]:
planets, orbiters = orbits.get_starting_bodies()

x0, mu, names = dynamics.build(planets, orbiters)
x0 = dynamics.bary_shift(x0, mu)

n_bodies = len(names)
n_massive = len(planets)


f = lambda x, t: dynamics.nbody_rhs(x, mu)

jacobian = linearize.jacobian(n_bodies)
J0 = jacobian(x0, mu)

In [3]:
ev, owners, mode_idx, zero_idx = linearize.modes(J0, n_massive)

print(f"{'body':12s} {'max Re(lam)':>12} {'max Im(lam)':>12}   (1/s)")
for b, name in enumerate(names):
	mine = [j for j in mode_idx if owners[j] == b]
	if not mine:
		print(f"{name:12s} {'none':>12} {'none':>12}")
		continue
	print(f"{name:12s} {max(abs(ev[mine].real)):12.3e} {max(abs(ev[mine].imag)):12.3e}")
print(f"\ntranslation zeros: |lam| <= {abs(ev[zero_idx]).max():.1e}")


body          max Re(lam)  max Im(lam)   (1/s)
Sun                  none         none
Venus           4.539e-07    3.209e-07
Earth           2.888e-07    2.042e-07
Mars            1.649e-07    1.166e-07
Jupiter         2.368e-08    1.674e-08
Saturn          9.591e-09    6.782e-09
LEO Orbiter     1.411e-03    9.977e-04
MEO Orbiter     2.044e-04    1.445e-04
GEO Orbiter     1.032e-04    7.294e-05

translation zeros: |lam| <= 1.8e-13


In [4]:
Rs = linearize.stiffness_ratio(ev)

mag = abs(ev[mode_idx])
fast, slow = mode_idx[np.argmax(mag)], mode_idx[np.argmin(mag)]

print(f"R_s (Lecture 13)  = {Rs:.1e}")
print(f"fastest |lam|     = {abs(ev[fast]):.3e} 1/s   ({names[owners[fast]]}, timescale {2 * np.pi / abs(ev[fast]) / 60:.1f} min)")
print(f"slowest |lam|     = {abs(ev[slow]):.3e} 1/s   ({names[owners[slow]]}, timescale {2 * np.pi / abs(ev[slow]) / 3.15576e7:.1f} yr)")
print(f"timescale spread  = {abs(ev[fast]) / abs(ev[slow]):.2e}")

R_s (Lecture 13)  = 2.2e+20
fastest |lam|     = 1.411e-03 1/s   (LEO Orbiter, timescale 74.2 min)
slowest |lam|     = 6.781e-09 1/s   (Saturn, timescale 29.4 yr)
timescale spread  = 2.08e+05


In [ ]:
lam = linearize.stability_eigs(ev[mode_idx])
P_fast = 2 * np.pi / max(abs(ev[mode_idx].imag)	)   # period of the fastest orbit

print(f"{'method':14s} {'largest stable dt':>18} {'steps per fastest orbit':>24}")
for name, amp in linearize.METHODS.items():
	dt_max = linearize.stable_dt(amp, lam)
	if dt_max == np.inf:
		print(f"{name:14s} {'no limit':>18} {'set by accuracy':>24}")
	elif dt_max == 0.0:
		print(f"{name:14s} {'none':>18} {'unstable at every dt':>24}")
	else:
		print(f"{name:14s} {dt_max:16.0f} s {P_fast / dt_max:24.1f}")

method          largest stable dt  steps per fastest orbit
forward Euler                none     unstable at every dt
RK4                        1972 s                      3.2
trapezoid                no limit          set by accuracy
BDF2                     no limit          set by accuracy
